In [1]:
# A CSP is problem is defined by variables, domains, and constraints that must be satisfied. 
# The goal is to assign a value to each variable so that all the constraints are satisfied. 

In [3]:
# Backtracking earch is a depth-first search method
# It explores possible solutions by making a series of decisions and backtracks when a constraint violation is detected
# Sudoku is a popular CSP problem

1. Start with an Empty assignment
2. Choose an unassigned variable
3. Assign a value to the variable that satisfies the constraints
4. If the assignment is consistent, move to the next variable
5. If the assignment violates any constraints, backtrack and try a new value
6. Repeat until all variables are assigned or no solution is found

# Limitations of Backtracking:
* Not efficient in large search space. It will be slow to find a solution
* Thrashing: Backtracking repeatedly trying the same value
* There is no guatantee that the solution will be optimal.

In [69]:
# Core CSP Data Structure
from __future__ import annotations
from typing import Any, Callable, Optional
from copy import deepcopy

class CSP:
    """
    The definition of a constraint satisfied problem

    * Variables are arbitrary hashable objects
    * The domain maps each variable to its comprehensive set of possible values
    * Neighbors maps each variable to the variables it shares constraints with
    * The constraint function evaluates the variables against the constraints
    """

    def __init__(self, variables: list, domains: dict[Any, list], neighbors: dict[Any,list], constraint: Callable[[Any, Any, Any, Any], bool]):
        self.variables = variables
        self.domains = domains          # {var: [val1, val2, ...]}
        self.neighbors = neighbors      # {var: [var1, var2, ...]}
        self.constraint = constraint

    def is_consistent(self, var, val, assignment: dict) -> bool:
        """
        Sudoku:
        Check for val against every assigned neighbor
        """
        for neighbor in self.neighbors[var]:
            if neighbor in assignment:
                if not self.constraint(var, val, neighbor, assignment[neighbor]):
                    return False
        return True

In [70]:
def make_sudoku_csp(grid: str) -> tuple[CSP, list]:
    """
    Build a CSP for a 9×9 Sudoku.
 
    grid: 81-character string, '0' or '.' for empty cells, '1'-'9' for givens.
    Returns (csp, variable_order) where each variable is an int 0–80
    (row-major: cell (r,c) = r*9+c).
    """
    cells = list(range(81))
 
    # Row, column, and box peers for each cell
    def peers(i):
        r, c = divmod(i, 9)
        box_r, box_c = (r // 3) * 3, (c // 3) * 3
        row_cells  = {r * 9 + cc for cc in range(9)}
        col_cells  = {rr * 9 + c  for rr in range(9)}
        box_cells  = {(box_r + dr) * 9 + (box_c + dc)
                      for dr in range(3) for dc in range(3)}
        return (row_cells | col_cells | box_cells) - {i}
 
    neighbors = {i: list(peers(i)) for i in cells}
 
    # Domains: given cells are fixed; empty cells can be 1-9
    domains = {}
    for i, ch in enumerate(grid.replace('.', '0')):
        domains[i] = [int(ch)] if ch != '0' else list(range(1, 10))
 
    # All-different constraint
    def constraint(v1, val1, v2, val2):
        return val1 != val2
 
    return CSP(cells, domains, neighbors, constraint)

In [71]:
def _restore(pruned: dict, domains: dict):
    for (var, val) in pruned:
        domains[var].append(val)

In [72]:
def _select_unassigned_variable(assignment: dict, domains: dict, csp: CSP):
    """
    MRV (Minimum Remaining Values): pick the unassigned variable whose
    current domain is smallest - i.e., the most constrained variable.
    Ties are broken by degree (most constraints with other unassigned vars)
    """
    unassigned = [v for v in csp.variables if v not in assignment]
    return min(unassigned, key=lambda v: (len(domains[v]), -sum(1 for n in csp.neighbors[v] if n not in assignment)))

In [73]:
def _order_domain_values(var, assignment: dict, domains: dict, csp: CSP):
    """
    LCV (:east Constraining Value): try values that rule out 
    the fewest choices for unassigned variables first
    """
    def count_conflicts(val):
        total = 0
        for neighbor in csp.neighbors[var]:
            if neighbor not in assignment:
                for nval in domains[neighbor]:
                    if not csp.constraint(var, val, neighbor, nval):
                        total += 1
        return total
    return sorted(domains[var], key=count_conflicts)

In [74]:
def _forward_check(var, val, assignment: dict, domains: dict, csp: CSP):
    """
    After assigning var=val, remove any domain value from unassigned neighbors
    that is now inconsistent.

    Returns (ok, pruned) where pruned is a dict of {(neighbor, val)} removedok=False means a domain was wiped out; caller must still restore pruned
    """
    pruned: dict[tuple, Any] = {}
    ok = True
    for neighbor in csp.neighbors[var]:
        if neighbor in assignment:
            continue
        for nval in list(domains[neighbor]):
            if not csp.constraint(neighbor, nval, var, val):
                domains[neighbor].remove(nval)
                pruned[(neighbor, nval)] = True
                if not domains[neighbor]:
                    ok = False
                    return ok, pruned # return early but pruned is intact
    return ok, pruned

In [75]:
# Backtacking Algo Implementation. AI - A Modern Approach Figure 5.5
def backtracking_search(csp: CSP) -> Optional[dict]:
    """
    An entry point. THis returns a complete assignment dict
    or None if unsolvable. Somains are deep-copied so the
    original is never mutated.
    """
    domains = deepcopy(csp.domains)
    return _backtrack({}, domains, csp)

def _backtrack(assignment: dict, domains: dict, csp: CSP) -> Optional[dict]:
    # Base case - every variable is assigned
    if len(assignment) == len(csp.variables):
        return assignment
    unassigned = _select_unassigned_variable(assignment, domains, csp)
    for val in _order_domain_values(unassigned, assignment, domains, csp):
        if csp.is_consistent(unassigned, val, assignment):
            assignment[unassigned] = val
            # FOrward checking - prune the domains of unassigned neighbors
            ok, pruned = _forward_check(unassigned, val, assignment, domains, csp)
            if ok:
                result = _backtrack(assignment, domains, csp)
                if result is not None:
                    return result
            del assignment[unassigned]
            _restore(pruned, domains)
    return None # trigger backtrack
        

In [76]:
def solve_sudoku(grid: str) -> Optional[str]:
    csp = make_sudoku_csp(grid)
    result = backtracking_search(csp)
    if result is None:
        return None
    return "".join(str(result[i]) for i in range(81))

 
def pretty_print(grid81: str):
    for r in range(9):
        row = [c if c != '0' else '.' for c in grid81[r * 9: r * 9 + 9]]
        print(" ".join(row[:3]), "|", " ".join(row[3:6]), "|", " ".join(row[6:]))
        if r in (2, 5):
            print("------+-------+------")
 

In [77]:
if __name__ == "__main__":
    # "Hardest" puzzle often used in benchmarks (Inkala 2010)
    puzzle = "800000000003600000070090200050007000000045700000100030001000068008500010090000400"
 
    print("Puzzle:")
    pretty_print(puzzle)
 
    solution = solve_sudoku(puzzle)
 
    if solution:
        print("\nSolution:")
        pretty_print(solution)
    else:
        print("No solution found.")


Puzzle:
8 . . | . . . | . . .
. . 3 | 6 . . | . . .
. 7 . | . 9 . | 2 . .
------+-------+------
. 5 . | . . 7 | . . .
. . . | . 4 5 | 7 . .
. . . | 1 . . | . 3 .
------+-------+------
. . 1 | . . . | . 6 8
. . 8 | 5 . . | . 1 .
. 9 . | . . . | 4 . .

Solution:
8 1 2 | 7 5 3 | 6 4 9
9 4 3 | 6 8 2 | 1 7 5
6 7 5 | 4 9 1 | 2 8 3
------+-------+------
1 5 4 | 2 3 7 | 8 9 6
3 6 9 | 8 4 5 | 7 2 1
2 8 7 | 1 6 9 | 5 3 4
------+-------+------
5 2 1 | 9 7 4 | 3 6 8
4 3 8 | 5 2 6 | 9 1 7
7 9 6 | 3 1 8 | 4 5 2


In [ ]:
 # Min-Conflicts Heuristic:
"""
With this algorithm, you essentially choose one of the objects at random.
Find the space in play where that object has the minimum amount of conflicts,
however they are defined. Continue to do this until all in-play objects have
no or the minimum possible amount of conflicts
"""
import random

def _count_conflicts(csp: CSP, var, val assignment: dict) -> int:
    """ Count the constraint violations for var=val """
    return sum(1 for neighbor in csp.neighbors[var] if not csp.constraint(var, val, neighbor, assignment[neighbor]))

def _conflicted_variables(csp: CSP, assignment: dect) -> list:
    """return all vars that violate at least one constraint"""
    return [v for v in csp.variablies if any(not csp.constraint(v, assignment[v], n, assignment[n]) for n in csp.neighbors[v])]

def _initial_assignment(csp: CSP) -> dict:
    """ Greedy start: assign each variable the value with the fewest current conflicts """
    assignment = {}
    for val in csp.variables:
        val = min(csp.domains[var], key=lambda v: _count_conflicts(csp, var, v, assignment))
        assignment[var] = val
    return assignment

def min_conflicts(csp: CSP, max_steps: int = 100_000) -> dict | None:
    """MIN-CONFLICTS local search - AI - A Modern Approach Figure 5.9"""
    current = _initial_assignment(csp)
    for _ in range(max_steps):
        conflicted = _conflicted_variables(csp, current)
        if not conflicted:
            return
        rand_var = random.choice(conflicted)
        min_conf = min(_count_conflicts(csp, rand_var, v, current) for v in csp.domains[var])
        best_vals = [v for v in csp.domains[rand_var] if _count_conflicts(csp, rand_var, v, current) == min_conf]
        current[rand_var] = random.choice(best_vals)
    return None





















































